In [1]:
!pip install -r requirements.txt

In [2]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

## Data Preprocessing

In this section we will transform the image into a matrix.

### Load the Images

In [3]:
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            images.append(img)
    return images

Each image can be represented as a matrix of pixels. The RGB format involves a complex setup of three matrices representing the intensity of each color channel. To simplify calculations and improve efficiency, we convert the image to grayscale, which works effectively for the following algorithms.

For an image represented as an $N \times M$ matrix, we flatten it into an $NM \times 1$ vector to execute subsequent vector operations.

In [4]:
def flatten_image(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    return image.flatten()

### Data Normalization

The next step for the PCA algorithm is data normalization. We define our data matrix $X$ as:

$$X = [x_1, x_2, x_3, \ldots, x_n] \tag{1}$$

where $x_i$ are the flattened image vectors from our dataset.

In [5]:
def create_dataset(folder):
    images = load_images_from_folder(folder)
    dataset = []
    for img in images:
        flat_img = flatten_image(img)
        dataset.append(flat_img)
    return np.array(dataset)

### Mean Vector

We then calculate the mean vector:

$$\mu = \frac{1}{n} \sum_{i=1}^{n} x_i \tag{2}$$

In [6]:
def calculate_mean_vector(X):
    mean_vector = np.mean(X, axis=0)
    return mean_vector

### Centralized Data

The centralized data of the set is then represented as:

$$\bar{x}_i = x_i - \mu \tag{3}$$

In [7]:
def centralize_data(X, mean_vector):
    X_centered = X - mean_vector
    return X_centered

### Covariance Matrix

The following step involves calculating the covariance matrix using the formula:

$$C = \frac{1}{n} \bar{X} \bar{X}^T \tag{4}$$

To optimize calculation for high-resolution images, we use a mathematical trick by defining a smaller matrix $C'$:

$$C' = \bar{X}^T \bar{X} \tag{5}$$

This allows us to process datasets where the number of images is much smaller than the number of pixels.

In [8]:
def calculate_covariance_matrix(X):
    # Center the data
    X_centered = X - calculate_mean_vector
    # Calculate the covariance matrix
    covariance_matrix = np.cov(X_centered, rowvar=False)

    return covariance_matrix

## Eigenvectors and Eigenvalues

To proceed, we must utilize Eigenvectors and Eigenvalues. By definition, the transformation of an eigenvector $v$ by a matrix $A$ is:

$$Av = \lambda v \tag{6}$$

where $\lambda$ represents the eigenvalue.

Assuming $C' = \bar{X}^T \bar{X}$ is our $A$:

$$(\bar{X}^T \bar{X}) v = \lambda v \tag{7}$$

Left-multiplying by $\bar{X}$:

$$(\bar{X} \bar{X}^T)(\bar{X} v) = \lambda (\bar{X} v) \tag{8}$$

This proves that $(\bar{X} v)$ is the eigenvector of the large covariance matrix $C$.

In [9]:
def define_eigenvectors(covariance_matrix):
    eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)
    return eigenvalues, eigenvectors

## Power Method and Projection

We use the power method to find the dominant eigenvector:

$$x_{k+1} = \frac{A x_k}{\| A x_k \|} \tag{9}$$

This iterates until convergence. We then build matrix $Y$ (the "eigendrone" space) and project our data:

$$Z = Y^T \bar{X} \tag{10}$$

For a test image $x_{\text{test}}$, we center and project it:

$$Z_{\text{test}} = Y^T (x_{\text{test}} - \mu) \tag{11}$$

Finally, we calculate the Euclidean distance between projections. If the distance is below an experimental threshold, the object is classified as a drone.

In [10]:
def power_method(covariance_matrix, num_iterations=30):
    n, _ = covariance_matrix.shape
    # Start with a random vector
    b_k = np.random.rand(n)

    for _ in range(num_iterations):
        # Calculate the matrix-by-vector product
        b_k1 = np.dot(covariance_matrix, b_k)

        # Normalize the resulting vector
        b_k1_norm = np.linalg.norm(b_k1)
        if b_k1_norm == 0:
            return b_k  # Return the current vector if it becomes zero
        b_k = b_k1 / b_k1_norm

    return b_k

### Testing

In [11]:
def z_test(X, mean_vector, eigenvector):
    # Project the data onto the eigenvector
    projections = np.dot(X - mean_vector, eigenvector)
    return projections

In [12]:
def calculate_euclidean_distance(projections, threshold):
    distances = np.linalg.norm(projections, axis=1)

    return distances < threshold